# Notebook 01: Blockchain and Cryptography Basics

Welcome to the first notebook in this series! In this lesson, we will explore the fundamental concepts that make blockchains secure and reliable: **Cryptography** and **Data Structuring**.

We will cover:
1. Generating private and public keys.
2. Signing and verifying messages.
3. Attempting to falsify messages (and why it fails).
4. Creating a simple, linked Blockchain structure.

### Prerequisites
Before we begin, we need to install the `cryptography` library. This allows us to perform RSA key generation and message signing.

In [ ]:
!pip install cryptography

## 1. Digital Signatures and Keys

In a blockchain, users need a way to prove they authored a specific transaction (message) without revealing their secret password. This is done using **Public-Key Cryptography**.

Every user has a pair of keys:
- **Private Key:** Kept secret. Used to *sign* messages.
- **Public Key:** Shared with everyone. Used to *verify* signatures.

Let's write some code to generate these keys.

In [ ]:
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.asymmetric import padding
from cryptography.hazmat.backends import default_backend
from cryptography.hazmat.primitives.asymmetric import rsa
from cryptography.hazmat.primitives import serialization

# Generate private key
private_key = rsa.generate_private_key(
    public_exponent=65537,
    key_size=2048,
    backend=default_backend()
)

# Getting public key from private key
public_key = private_key.public_key()

print("Keys generated successfully!")

## 2. Signing and Verifying a Message

Now that we have our keys, let's say Nelson wants to send the message `"Nelson likes cat"`.
He uses his private key to create a unique **signature** for this exact message.

In [ ]:
MESSAGE = b"Nelson likes cat"

# Nelson signs the message with his Private Key
signature = private_key.sign(
    MESSAGE,
    padding.PSS(
        mgf=padding.MGF1(hashes.SHA256()),
        salt_length=padding.PSS.MAX_LENGTH
    ),
    hashes.SHA256()
)

print(f"Original Message: {MESSAGE}")
print(f"Signature (first 50 bytes): {signature[:50]}...")

Now, anyone else can use Nelson's **Public Key** to verify that this signature genuinely came from Nelson and matches the message.

In [ ]:
# The network verifies the signature using Nelson's Public Key
try:
    public_key.verify(
        signature,
        MESSAGE,
        padding.PSS(
            mgf=padding.MGF1(hashes.SHA256()),
            salt_length=padding.PSS.MAX_LENGTH
        ),
        hashes.SHA256()
    )
    print("Success! The signature is valid. Nelson definitely wrote this message.")
except Exception as e:
    print("Verification Failed!")

## 3. Attempting to Falsify a Message

What happens if someone tries to change the message to `"Nelson hates cat"` but uses the same signature? Or if they just make up a fake signature?

Let's see what happens when verification fails.

In [ ]:
fake_message = b'Nelson hates cat'
fake_signature = b'Fake Signature'

try:
    public_key.verify(
        fake_signature,
        fake_message,
        padding.PSS(mgf=padding.MGF1(hashes.SHA256()),
                    salt_length=padding.PSS.MAX_LENGTH),
        hashes.SHA256()
    )
    print("Success! (Wait, this shouldn't happen...)")
except Exception as e:
    print("Verification Failed! The cryptography library rejected the fake message/signature.")
    print(f"Error: {type(e).__name__}")

## 4. Building a Simple Blockchain

A blockchain is a linked list of data structures (blocks). Each block contains data (like our messages) and, crucially, a **cryptographic hash** of the block that came before it.

This creates a chain. If someone tries to change data in Block A, the hash of Block A changes, which breaks the link to Block B.

Let's model this in Python.

In [ ]:
import hashlib
import json

class Block:
    id = None
    history = None
    parent_id = None
    parent_hash = None
    
    def to_dict(self):
        return {
            'id': self.id,
            'history': self.history,
            'parent_id': self.parent_id,
            'parent_hash': self.parent_hash
        }

    def calculate_hash(self):
        # Convert the block's data to a JSON string, encode it to bytes, and hash it
        block_string = json.dumps(self.to_dict(), sort_keys=True).encode('utf-8')
        return hashlib.sha256(block_string).hexdigest()

# Create the Genesis Block (the first block)
block_A = Block()
block_A.id = 1
block_A.history = 'Nelson likes cat'

print(f"Block A created. Hash: {block_A.calculate_hash()}")

# Create Block B, linked to Block A
block_B = Block()
block_B.id = 2
block_B.history = 'Marie likes dog'
block_B.parent_id = block_A.id
block_B.parent_hash = block_A.calculate_hash()

print(f"Block B created. Parent Hash matches Block A: {block_B.parent_hash == block_A.calculate_hash()}")

# Create Block C, linked to Block B
block_C = Block()
block_C.id = 3
block_C.history = 'Sky likes bird'
block_C.parent_id = block_B.id
block_C.parent_hash = block_B.calculate_hash()

print(f"Block C created. Linking to Block B hash: {block_C.parent_hash}")

### Tampering with the Blockchain
If we change `block_A.history`, `block_A`'s hash will change. Because `block_B` saved the original hash, the chain becomes broken and the tamper is detected immediately!